## SMPL : A Skinned Multi-Person Linear Model(2015)

### 용어 정리
* skinned vertex : 관절과 같은 뼈의 움직임에 따라 변형되는 정점
* taffy effect : 인접한 뼈들의 영향이 지나치게 클 경우, 해당 관절이 늘어나게 되는 현상
* bowtie effect : 정점이 두 개의 뼈에 겹쳐서 뾰족하게 변형되어 '넥타이' 모양이 되는 현상

### 1. Introduction

#### 1) 연구목표
* 다양한 신체 형태를 표현하고 자연스로운 자세에 따른 변형을 취할 수 있으며 실제 인간과 같은 연조직 동작을 나타낼 수 있는 현실적인 애니메이션 인체를 만드는 것

#### 2) problem
* 대부분의 현실적 모델들은 데이터에서 학습되지만, 렌더링 엔진과 호환되지 않음
    * $\rightarrow$ SMPL은 기존 렌더링 엔진과 호환 가능

### 2. Related Work
* Blend Skinning : 스켈레톤(뼈대)의 움직임을 3D 메시 표면에 전달하는 기술(skeleton이 움직이면 vertex도 영향을 받아 움직임) $\rightarrow$ linear blend skinning(LBS) 모델 'taffy', 'bowtie' 효과 존재

* Auto-rigging : 3D 메시에 스켈레톤을 자동으로 생성하고 설정하는 과정 $\rightarrow$ LBS모델의 문제를 해결해주지는 않음
* blend-shape : 미리 만든 여러 형태들을 섞어서 새로운 변형을 만드는 기술 $\rightarrow$ 단일 자세/단일 형상 모델에만 수행가능 
* learning pose and shape models : 초기 방법들은 PCA를 사용해 인간의 체형 공간을 특성화하지만 체형이 포즈와 함께 어떻게 변하는지 모델링하지는 않음. SCAPE나 삼각형 변형을 사용하여 다양한 체형과 자세를 표현 $\rightarrow$ 하지만 구현 복잡도와 계산비용이 크고 pose 변화와 shape변화가 비선형적으로 뒤섞여 처리가 어려움
  
### 3. Model Formulation


#### 1) SMPL
* LBS기반으로 shape 파라미터 $\beta$와 pose 파라미터 $\theta$를 선형으로 융합해 다양한 사람의 자세 + 체형을 하나의 일관된 mesh 로 재현
* 정점 기반 skinning, shape와 blend 각각 blend shape를 적용(PCA기반으로)

#### 파이프라인

* 1단계: 기본 템플릿 메시 (T̄)

   ↓

* 2단계: Shape blend shapes 적용
   * β (체형 파라미터) → 개인 체형 생성

   ↓

  
* 3단계: Pose blend shapes 적용
   * θ (포즈 파라미터) → 포즈에 따른 교정 변형 추가

   ↓

  
* 4단계: Blend Skinning (LBS) 적용
   * 관절 회전으로 실제 포즈 적용

   ↓

  
* 5단계: 최종 메시



<img src = "../assets/smpl-1.png">

* $M(\theta, \beta) = W(Tp(\beta, \theta), J(\beta), \theta, \mathcal{W}) : \mathbb{R}^{|\theta| \times |\beta|} \rightarrow \mathbb{R}^{3N}$

* $Tp(\beta, \theta) = \bar{\mathbf{T}} + Bs(\beta) + Bp(\theta)$

* **Shape blend shapes**
* $Bs(\beta; \mathcal{S}) = \sum^{|\beta|}_{n=1}\beta_n\mathbf{S}_n : \mathbb{R}^{|\beta|} \rightarrow \mathbb{R}^{3N}$ $\rightarrow S_n$ : PCA를 통해 학습된 n번째 shape blend shape(주성분)

* 사람(3D 신체 스캔)마다 : 6890 vertices x 3(x,y,x) = 20670차원 

* **Joint locations**
* $J(\beta; \mathcal{J},\bar{\mathbf{T}},\mathcal{S}) = \mathcal{J}(\bar{\mathbf{T}} + Bs(\beta;\mathcal{S}) : \mathbb{R}^{|\beta|} \rightarrow \mathbb{R}^{3K}$

* **Pose blend shape**
* $Bp(\theta; \mathcal{P}) = \sum^{9K}_{n=1}(R_n(\theta) - R_n(\theta^*))\mathbf{P}_n : \mathbb{R}^{|\theta|} \rightarrow \mathbb{R}^{3N}$ $\rightarrow P_n$ : PCA를 통해 학습된 n번째 pose blend shape(주성분)


* 고정된 N = 6890개의 vertex
* 고정된 K = 23개의 joint
* $\beta$ : shape parameter (10차원 벡터)
* $\theta$ : pose parameter (23 x 3 = 69차원 벡터)
---
* $\mathcal{W} \in \mathbb{R}^{N \times K}$ : a set of blend weights
* $\bar{\mathbf{T}} \in \mathbb{R}^{3N}$ : 기준 자세에 있는 중립형 사람의 mesh(mean template shpe in the zero pose($\theta^*$)


### 4. Training

#### 1) 데이터
* 3D scanner 로 전신 스캔한 데이터 수집
* male : 1700, female : 2100

<img src = "../assets/smpl-2.png">

#### 2) template 정렬(Non-rigid Registration)
* 모든 scan 데이터를 동일한 위상(topology)의 템플릿 메쉬와 정합해야함
#### 3) shape blend shapes 추출
* 다른 사람들의 중립 자세로 normalize 한 뒤에 평균 모델에서 편차를 PCA등으로 분해해 shape basis를 학습

### 5. 평가

#### 1) SCAPE
* SCAPE는 비선형 deformation Model이고 SMPL은 Linearity를 강하게 활용 $\rightarrow$ 학습 안정성, 구현 단순성
* 논문에서 다양한 체형과 자세 샘플링 결과, SMPL이 유사하거나 더 나은 품질

### 6. 장점 및 한계

#### 1) 장점
* 통합적인(shape+pose) 파라메트릭 모델(직관적, 선형결합 구조로 계산 간소화)
* LBS메커니즘 사용으로, 기존 그래픽 엔진등과 쉽게 통합 가능
* 기존 SCAPE 보다 파이프라인 간단하고 계산량도 적음
* 컴퓨터 비전에서 2D 이미지에서 $\theta, \: \beta$를 추정해 3D 재구성하는데 응용 가능
* 모든 사람과 모든 자세에 동일한 vertex 인덱스를 공유

#### 2) 한계
* 극단적 체형(임산부, 매우 뚱뚱하거나 마른 경우) PCA 기반 shape basis가 잘 표현하지 못함
* 실제 근육수축/이완을 완전히 재현하기에는 제한
* 기본모델은 얼굴, 손가락, 발가락의 디테일이 낮음 $\rightarrow$ 후속연구로 SMPL-X, SMPL+H등에서 세분화 모델
* 동적변화 : 움직일 때 살이 흔들리는 등 동적 물리 효과는 SMPL이 직접적으로 모델링하지 않음